# Scaling Test Data Generator

Generates the `Data/Test_*/` CSVs (time-course + initial-conc-vs-final-conc
sweep) used by the reaction-count/free-param scaling benchmark configs under
`Results/GPU Scaling Tests/`. Split out from `ODE Runner/run_model.ipynb`
(which stays focused on general single-network exploration) since this
notebook is specifically in service of the Bayesian-inference scaling study.

Each "Nested Test System" cell below is a genuinely reduced reaction network
(built from an explicit list of individual enzyme YAML files, not the full
`EC_FAS_ME1` folder with other enzymes zeroed out), cumulative one enzyme at
a time along the real FAS elongation/termination pathway. Every cell rebuilds
`network`/`species`/`sp`/`theta` for its own reduced system, then exports a
timeseries + endpoint sweep to `Data/Test_*/`, matching the corresponding
`Calculation Files/Test_*/` module used at inference time.

*Annette Thompson · Fox and Shirts Labs, CU Boulder · 2026*

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../")

from pathlib import Path
import re
import os
import json

import numpy as np
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import diffrax as dfrx
import pandas as pd

from Utilities.reaction_model_builder import (
    build_ode_system_from_reactions,
    make_namespace,
    set_scaling_group_values
)

In [2]:
def run_odes(time_range, y0, theta_run, max_steps=1_500, save_steps=True):

    sol = dfrx.diffeqsolve(
        dfrx.ODETerm(network),
        dfrx.Kvaerno5(),
        t0=time_range[0], t1=time_range[1], dt0=1e-6,
        y0=jnp.asarray(y0, dtype=jnp.float64),
        args=theta_run,
        saveat=dfrx.SaveAt(steps=True) if save_steps else dfrx.SaveAt(t1=True),
        stepsize_controller=dfrx.PIDController(
            rtol=1e-5, atol=1e-8, pcoeff=0.2, icoeff=0.4, dcoeff=0,
        ),
        max_steps=max_steps,
        throw=False,
    )

    num_tot_steps = int(np.asarray(sol.stats['num_steps']))

    # Anything needing max_steps (or that failed outright) is treated as a bad
    # set of conditions rather than a crash -- callers should skip/handle a (None, None).
    if num_tot_steps >= max_steps or bool(sol.result != dfrx.RESULTS.successful):
        print(f"Solver did not converge within {max_steps} steps.")
        return None, None

    num_ac_steps = int(np.asarray(sol.stats['num_accepted_steps']))

    if save_steps:
        T = sol.ts[:num_ac_steps]
        C = sol.ys[:num_ac_steps, :]
    else:
        T = sol.ts
        C = sol.ys if sol.ys.ndim == 2 else sol.ys[None, :]

    print(f"{num_ac_steps} accepted steps, {num_tot_steps} total steps.")

    return T, C

## Export to experiment csvs

Helpers to make timeseries and endpoint CSVs.

In [14]:
# Functions to make timeseries and endpoint CSVs

def choose_species_indices(selected_species=(), species_pattern=None):
    if species_pattern:
        pattern = re.compile(species_pattern)
        selected_names = [sp_name for sp_name in species if pattern.fullmatch(sp_name)]
    else:
        selected_names = list(selected_species)

    missing_species = [sp_name for sp_name in selected_names if sp_name not in species]
    if missing_species:
        raise ValueError(f"Requested species not found in reaction network: {missing_species}")

    if not selected_names:
        raise ValueError("No species selected for export.")

    return [getattr(sp, sp_name) for sp_name in selected_names]


def export_species_time_course(time_range, y0_run, theta_run, target_species_indices, folder_name, save_csv=True, n_points=11, min_gap_seconds=1):

    T, C = run_odes(time_range, y0_run, theta_run)

    target_species_indices = [int(idx) for idx in target_species_indices]
    species_data = C[:, target_species_indices]
    idx_to_name = {v: k for k, v in vars(sp).items()}
    target_names = [idx_to_name.get(target_species_idx, f"Species_{target_species_idx}") for target_species_idx in target_species_indices]

    columns = ["Time (s)"] + [f"{target_name} (uM)" for target_name in target_names]
    df = pd.DataFrame(np.column_stack([T, species_data]), columns=columns)

    T_arr = np.asarray(T)
    valid_idx = [0]
    for i in range(1, len(T_arr)):
        if T_arr[i] - T_arr[valid_idx[-1]] >= min_gap_seconds:
            valid_idx.append(i)
    valid_idx = np.array(valid_idx)

    n_keep = min(n_points, len(valid_idx))
    pick_positions = np.unique(np.round(np.linspace(0, len(valid_idx) - 1, n_keep)).astype(int))
    keep_rows = valid_idx[pick_positions]
    df_filtered = df.iloc[keep_rows].reset_index(drop=True)

    filename = f"../Data/{folder_name}/time_vs_conc.csv"
    if save_csv:
        df_filtered.to_csv(filename, index=False)
        print(f"Saved to {filename}")

    print("Run complete!")
    return df_filtered

def _round_sigfigs(value, sigfigs=1):
    value = float(value)
    if value == 0.0:
        return 0.0
    return float(f"{value:.{sigfigs}g}")


def _log_distance(output_a, output_b):
    """Max per-species distance in log10 space between two output vectors.

    Concentrations span many orders of magnitude, so a plain (relative)
    difference would be dominated by whichever species happens to be
    largest; comparing in log space treats an order-of-magnitude shift in
    ANY tracked species as equally significant.
    """
    eps = 1e-300
    log_a = np.log10(np.abs(np.asarray(output_a, dtype=np.float64)) + eps)
    log_b = np.log10(np.abs(np.asarray(output_b, dtype=np.float64)) + eps)
    return float(np.max(np.abs(log_a - log_b)))


def sweep_final_conc(sweep_lists, target_species, y0_run, theta_run, folder_name, time_range=(0, 720), max_steps=1000, save_csv=True, n_required=9, min_log_diff=0.2):
    """One ODE solve per column of `sweep_lists` ({species_name: [values]}), recording the
    final concentration of every species in `target_species`. A candidate row is kept only
    if its output differs from every already-kept row by at least min_log_diff orders of
    magnitude (max over target species) -- this keeps the sweep rows from clustering near
    each other in output space, so the n_required kept rows actually span different
    dynamical regimes instead of just being 9 similar-looking successes. Saves
    ../Data/<folder_name>/init_vs_final_conc.csv."""
    if len({len(v) for v in sweep_lists.values()}) != 1:
        raise ValueError("All sweep lists must have the same length.")

    idx_to_name = {v: k for k, v in vars(sp).items()}
    target_names = [idx_to_name.get(idx, f"Species_{idx}") for idx in target_species]

    rows = []
    kept_outputs = []
    for combo in zip(*sweep_lists.values()):
        y0_row = y0_run.copy()
        for name, conc in zip(sweep_lists, combo):
            y0_row[getattr(sp, name)] = _round_sigfigs(conc)
        _, C = run_odes(list(time_range), y0_row, theta_run, max_steps, save_steps=False)
        if C is None:
            print(f"Skipping bad-condition row: {dict(zip(sweep_lists, [_round_sigfigs(c) for c in combo]))}")
            continue

        output = [float(C[-1, idx]) for idx in target_species]
        if kept_outputs and min(_log_distance(output, prev) for prev in kept_outputs) < min_log_diff:
            print(f"Skipping too-similar row (< {min_log_diff} log10 units from an already-kept row): "
                  f"{dict(zip(sweep_lists, [_round_sigfigs(c) for c in combo]))}")
            continue

        rows.append([_round_sigfigs(c) for c in combo] + output)
        kept_outputs.append(output)
        if len(rows) >= n_required:
            break

    if len(rows) < n_required:
        raise RuntimeError(
            f"Only found {len(rows)} successful, sufficiently-different sweep conditions out of "
            f"{n_required} required. Try a smaller min_log_diff or a wider/larger candidate factor list."
        )

    df_sweep = pd.DataFrame(
        rows,
        columns=[f"{name} (uM)" for name in sweep_lists] + [f"{name} (uM)" for name in target_names],
    )
    if save_csv:
        filename = f"../Data/{folder_name}/init_vs_final_conc.csv"
        df_sweep.to_csv(filename, index=False)
        print(f"Saved to {filename}")
    return df_sweep


_SWEEP_UP = np.geomspace(1.05, 100.0, 30)
_SWEEP_DOWN = np.geomspace(0.95, 0.01, 30)
SWEEP_FACTORS = tuple(float(f) for pair in zip(_SWEEP_UP, _SWEEP_DOWN) for f in pair)


def make_nonuniform_sweep(y0_run, names, factors=SWEEP_FACTORS, offsets=(-0.08, -0.04, 0.0, 0.04, 0.08)):
    """Build a sweep_final_conc-ready dict with non-uniform row scaling.

    Each row uses a shared base factor, but every species gets a small, deterministic
    species-specific offset so multipliers are not uniform across species.
    """
    if not names:
        raise ValueError("names must contain at least one species")

    sweep = {name: [] for name in names}
    n_offsets = len(offsets)

    for row_idx, factor in enumerate(factors):
        for name_idx, name in enumerate(names):
            base = float(y0_run[getattr(sp, name)])
            offset = offsets[(row_idx + 2 * name_idx) % n_offsets]
            value = base * factor * (1.0 + offset)
            sweep[name].append(_round_sigfigs(value))

    return sweep

## Nested Test Systems

Sampler-benchmark test systems, each a genuinely reduced reaction network
(loaded from an explicit list of individual enzyme YAML files, not the full
`EC_FAS_ME1` folder with other enzymes zeroed out). Each cell rebuilds
`network`/`species`/`sp`/`theta` for its own reduced system and tracks only
its system's final-output species (matching the corresponding
`Calculation Files/Test_*/` module), then exports a timeseries + endpoint
sweep to `Data/Test_*/`.

**Note:** each cell overwrites the global `network`/`species`/`sp`/`theta`
from the previous one -- run them independently (or top-to-bottom) rather
than relying on state from a skipped cell.

In [15]:
# Nested Test System 1: FabD

enzyme_name = "Test_FabD"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [Path('../Reactions/EC_FAS_ME1/FabD.yaml')]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 0.001
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ("C3_MalACP",)
generated_species_pattern = None

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates, 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "C3_MalCoA", "ACP"]),
    target_species, y0, theta, enzyme_name, max_steps=80,
)

61 accepted steps, 61 total steps.
Saved to ../Data/Test_FabD/time_vs_conc.csv
Run complete!
61 accepted steps, 61 total steps.
60 accepted steps, 60 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.0009, 'C3_MalCoA': 500.0, 'ACP': 9.0}
62 accepted steps, 62 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.001, 'C3_MalCoA': 700.0, 'ACP': 10.0}
58 accepted steps, 58 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.0008, 'C3_MalCoA': 400.0, 'ACP': 8.0}
66 accepted steps, 66 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.002, 'C3_MalCoA': 700.0, 'ACP': 10.0}
55 accepted steps, 55 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.0006, 'C3_MalCoA': 300.0, 'ACP': 7.0}
66 accepted steps, 66 total steps.
55 accepted steps, 55 total steps.
66 accepted steps, 66 total

In [16]:
# Nested Test System 2: FabD + FabH

enzyme_name = "Test_FabD_FabH"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 0.005
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10
y0[sp.FabH] = 0.005
y0[sp.C2_AcCoA] = 500

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ("C4_BKeAcACP",)
generated_species_pattern = None

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates, 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=100
)

83 accepted steps, 83 total steps.
Saved to ../Data/Test_FabD_FabH/time_vs_conc.csv
Run complete!
83 accepted steps, 83 total steps.
83 accepted steps, 83 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.005, 'FabH': 0.005, 'C3_MalCoA': 400.0, 'ACP': 10.0, 'C2_AcCoA': 500.0}
87 accepted steps, 87 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.006, 'FabH': 0.007, 'C3_MalCoA': 600.0, 'ACP': 10.0, 'C2_AcCoA': 600.0}
79 accepted steps, 79 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.004, 'FabH': 0.004, 'C3_MalCoA': 400.0, 'ACP': 9.0, 'C2_AcCoA': 400.0}
90 accepted steps, 90 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.008, 'FabH': 0.007, 'C3_MalCoA': 700.0, 'ACP': 10.0, 'C2_AcCoA': 700.0}
76 accepted steps, 76 total steps.
91 accepted steps, 91 total steps.
75 accepted steps, 75 total steps.
Skippin

In [17]:
# Nested Test System 3: FabD + FabH + FabG

enzyme_name = "Test_FabD_FabH_FabG"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabG.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 0.01
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10
y0[sp.FabH] = 0.01
y0[sp.C2_AcCoA] = 500
y0[sp.FabG] = 0.01
y0[sp.NADPH] = 1000

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ("C4_BHyAcACP",)
generated_species_pattern = None

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates (cofactor NADPH held fixed), 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "FabG", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=110
)

89 accepted steps, 89 total steps.
Saved to ../Data/Test_FabD_FabH_FabG/time_vs_conc.csv
Run complete!
89 accepted steps, 89 total steps.
88 accepted steps, 88 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.009, 'FabH': 0.01, 'FabG': 0.009, 'C3_MalCoA': 500.0, 'ACP': 10.0, 'C2_AcCoA': 500.0}
90 accepted steps, 90 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.01, 'FabH': 0.01, 'FabG': 0.01, 'C3_MalCoA': 600.0, 'ACP': 10.0, 'C2_AcCoA': 600.0}
85 accepted steps, 85 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.008, 'FabH': 0.007, 'FabG': 0.008, 'C3_MalCoA': 400.0, 'ACP': 8.0, 'C2_AcCoA': 400.0}
94 accepted steps, 94 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.02, 'FabH': 0.01, 'FabG': 0.01, 'C3_MalCoA': 700.0, 'ACP': 10.0, 'C2_AcCoA': 800.0}
82 accepted steps, 82 total steps.
Skipping too-simila

In [18]:
# Nested Test System 4: FabD + FabH + FabG + FabZ

enzyme_name = "Test_FabD_FabH_FabG_FabZ"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabG.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabZ.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 0.05
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10
y0[sp.FabH] = 0.05
y0[sp.C2_AcCoA] = 500
y0[sp.FabG] = 0.05
y0[sp.NADPH] = 1000
y0[sp.FabZ] = 0.05

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ("C4_EnAcACP",)
generated_species_pattern = None

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates (cofactor NADPH held fixed), 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "FabG", "FabZ", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=130
)

109 accepted steps, 109 total steps.
Saved to ../Data/Test_FabD_FabH_FabG_FabZ/time_vs_conc.csv
Run complete!
109 accepted steps, 109 total steps.
109 accepted steps, 109 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.05, 'FabH': 0.05, 'FabG': 0.04, 'FabZ': 0.05, 'C3_MalCoA': 500.0, 'ACP': 9.0, 'C2_AcCoA': 500.0}
113 accepted steps, 113 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.06, 'FabH': 0.07, 'FabG': 0.06, 'FabZ': 0.06, 'C3_MalCoA': 600.0, 'ACP': 10.0, 'C2_AcCoA': 700.0}
106 accepted steps, 106 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.04, 'FabH': 0.04, 'FabG': 0.04, 'FabZ': 0.04, 'C3_MalCoA': 400.0, 'ACP': 8.0, 'C2_AcCoA': 400.0}
112 accepted steps, 112 total steps.
102 accepted steps, 102 total steps.
116 accepted steps, 116 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.08, 'FabH':

In [19]:
# Nested Test System 5: FabD + FabH + FabG + FabZ + FabI

enzyme_name = "Test_FabD_FabH_FabG_FabZ_FabI"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabG.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabZ.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabI.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 0.1
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10
y0[sp.FabH] = 0.1
y0[sp.C2_AcCoA] = 500
y0[sp.FabG] = 0.1
y0[sp.NADPH] = 1000
y0[sp.FabZ] = 0.1
y0[sp.FabI] = 0.1
y0[sp.NADH] = 1000

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ("C4_AcACP",)
generated_species_pattern = None

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates (cofactors held fixed), 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "FabG", "FabZ", "FabI", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=140
)

119 accepted steps, 119 total steps.
Saved to ../Data/Test_FabD_FabH_FabG_FabZ_FabI/time_vs_conc.csv
Run complete!
120 accepted steps, 120 total steps.
118 accepted steps, 118 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.09, 'FabH': 0.1, 'FabG': 0.09, 'FabZ': 0.1, 'FabI': 0.1, 'C3_MalCoA': 500.0, 'ACP': 10.0, 'C2_AcCoA': 400.0}
120 accepted steps, 120 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.1, 'FabH': 0.1, 'FabG': 0.1, 'FabZ': 0.1, 'FabI': 0.1, 'C3_MalCoA': 600.0, 'ACP': 10.0, 'C2_AcCoA': 600.0}
115 accepted steps, 115 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.08, 'FabH': 0.07, 'FabG': 0.08, 'FabZ': 0.09, 'FabI': 0.08, 'C3_MalCoA': 400.0, 'ACP': 7.0, 'C2_AcCoA': 400.0}
123 accepted steps, 123 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.2, 'FabH': 0.1, 'FabG': 0.1, 'FabZ': 0.1, 'Fa

In [20]:
# Nested Test System 6: FabD + FabH + FabG + FabZ + FabI + TesA

enzyme_name = "Test_FabD_FabH_FabG_FabZ_FabI_TesA"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabG.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabZ.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabI.yaml'),
    Path('../Reactions/EC_FAS_ME1/TesA.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 0.5
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10
y0[sp.FabH] = 0.5
y0[sp.C2_AcCoA] = 500
y0[sp.FabG] = 0.5
y0[sp.NADPH] = 1000
y0[sp.FabZ] = 0.5
y0[sp.FabI] = 0.5
y0[sp.NADH] = 1000
y0[sp.TesA] = 5

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ("C4_FA",)
generated_species_pattern = None

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates (cofactors held fixed), 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "FabG", "FabZ", "FabI", "TesA", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=150
)

125 accepted steps, 127 total steps.
Saved to ../Data/Test_FabD_FabH_FabG_FabZ_FabI_TesA/time_vs_conc.csv
Run complete!
122 accepted steps, 122 total steps.
128 accepted steps, 133 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.5, 'FabH': 0.5, 'FabG': 0.4, 'FabZ': 0.5, 'FabI': 0.5, 'TesA': 5.0, 'C3_MalCoA': 500.0, 'ACP': 9.0, 'C2_AcCoA': 500.0}
124 accepted steps, 124 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.6, 'FabH': 0.7, 'FabG': 0.6, 'FabZ': 0.6, 'FabI': 0.6, 'TesA': 6.0, 'C3_MalCoA': 700.0, 'ACP': 10.0, 'C2_AcCoA': 600.0}
122 accepted steps, 123 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.4, 'FabH': 0.4, 'FabG': 0.4, 'FabZ': 0.4, 'FabI': 0.4, 'TesA': 4.0, 'C3_MalCoA': 400.0, 'ACP': 8.0, 'C2_AcCoA': 400.0}
Solver did not converge within 150 steps.
Skipping bad-condition row: {'FabD': 0.8, 'FabH': 0.7, 'FabG': 0.7, 'FabZ': 0.7, 'FabI

In [23]:
# Nested Test System 7: FabD + FabH + FabG + FabZ + FabI + TesA + FabF

enzyme_name = "Test_FabD_FabH_FabG_FabZ_FabI_TesA_FabF"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabG.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabZ.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabI.yaml'),
    Path('../Reactions/EC_FAS_ME1/TesA.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabF.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 1
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10
y0[sp.FabH] = 1
y0[sp.C2_AcCoA] = 500
y0[sp.FabG] = 1
y0[sp.NADPH] = 1000
y0[sp.FabZ] = 1
y0[sp.FabI] = 1
y0[sp.NADH] = 1000
y0[sp.TesA] = 10
y0[sp.FabF] = 1

# New scaling groups introduced by FabF: x1, x3 (b1, b3, c2, e, a2 reused).
SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ()
generated_species_pattern = r"^C(\d+)_FA$"

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates (cofactors held fixed), 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "FabG", "FabZ", "FabI", "TesA", "FabF", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=200
)

170 accepted steps, 170 total steps.
Saved to ../Data/Test_FabD_FabH_FabG_FabZ_FabI_TesA_FabF/time_vs_conc.csv
Run complete!
171 accepted steps, 171 total steps.
169 accepted steps, 169 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 0.9, 'FabH': 1.0, 'FabG': 0.9, 'FabZ': 0.9, 'FabI': 1.0, 'TesA': 9.0, 'FabF': 1.0, 'C3_MalCoA': 400.0, 'ACP': 10.0, 'C2_AcCoA': 500.0}
173 accepted steps, 174 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 1.0, 'FabH': 1.0, 'FabG': 1.0, 'FabZ': 1.0, 'FabI': 1.0, 'TesA': 10.0, 'FabF': 1.0, 'C3_MalCoA': 600.0, 'ACP': 10.0, 'C2_AcCoA': 600.0}
165 accepted steps, 165 total steps.
171 accepted steps, 171 total steps.
162 accepted steps, 162 total steps.
Solver did not converge within 200 steps.
Skipping bad-condition row: {'FabD': 2.0, 'FabH': 2.0, 'FabG': 2.0, 'FabZ': 2.0, 'FabI': 2.0, 'TesA': 20.0, 'FabF': 2.0, 'C3_MalCoA': 800.0, 'ACP': 20.0, 'C2_AcCoA': 900.0}
157 acc

In [25]:
# Nested Test System 8: FabD + FabH + FabG + FabZ + FabI + TesA + FabF + FabA

enzyme_name = "Test_FabD_FabH_FabG_FabZ_FabI_TesA_FabF_FabA"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabG.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabZ.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabI.yaml'),
    Path('../Reactions/EC_FAS_ME1/TesA.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabF.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabA.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.FabD] = 1
y0[sp.C3_MalCoA] = 500
y0[sp.ACP] = 10
y0[sp.FabH] = 1
y0[sp.C2_AcCoA] = 500
y0[sp.FabG] = 1
y0[sp.NADPH] = 1000
y0[sp.FabZ] = 1
y0[sp.FabI] = 1
y0[sp.NADH] = 1000
y0[sp.TesA] = 10
y0[sp.FabF] = 1
y0[sp.FabA] = 1

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ()
generated_species_pattern = r"^C(\d+)_FA$"

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all system enzymes + substrates (cofactors held fixed), 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "FabG", "FabZ", "FabI", "TesA", "FabF", "FabA", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=190
)

166 accepted steps, 166 total steps.
Saved to ../Data/Test_FabD_FabH_FabG_FabZ_FabI_TesA_FabF_FabA/time_vs_conc.csv
Run complete!
166 accepted steps, 166 total steps.
169 accepted steps, 171 total steps.
167 accepted steps, 167 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 1.0, 'FabH': 1.0, 'FabG': 1.0, 'FabZ': 1.0, 'FabI': 1.0, 'TesA': 10.0, 'FabF': 1.0, 'FabA': 1.0, 'C3_MalCoA': 600.0, 'ACP': 10.0, 'C2_AcCoA': 600.0}
160 accepted steps, 160 total steps.
Solver did not converge within 190 steps.
Skipping bad-condition row: {'FabD': 2.0, 'FabH': 1.0, 'FabG': 1.0, 'FabZ': 1.0, 'FabI': 1.0, 'TesA': 20.0, 'FabF': 1.0, 'FabA': 1.0, 'C3_MalCoA': 700.0, 'ACP': 10.0, 'C2_AcCoA': 800.0}
157 accepted steps, 157 total steps.
179 accepted steps, 179 total steps.
153 accepted steps, 153 total steps.
180 accepted steps, 180 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 2.0, 'FabH': 2.0, 'FabG': 2.0, 'FabZ'

In [26]:
# Nested Test System 9: FabD + FabH + FabG + FabZ + FabI + TesA + FabF + FabA + FabB

enzyme_name = "Test_FullFAS"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

REACTIONS_PATH_TEST = [
    Path('../Reactions/EC_FAS_ME1/FabD.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabH.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabG.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabZ.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabI.yaml'),
    Path('../Reactions/EC_FAS_ME1/TesA.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabF.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabA.yaml'),
    Path('../Reactions/EC_FAS_ME1/FabB.yaml'),
]
network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH_TEST)
sp = make_namespace(species)
theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.C3_MalCoA] = 500.0
y0[sp.C2_AcCoA]  = 500.0
y0[sp.ACP]       = 10.0
y0[sp.NADPH]     = 1000.0
y0[sp.NADH]      = 1000.0

enzyme_concs = {
    sp.FabD: 1, sp.FabH: 1, sp.FabG: 1, sp.FabZ: 1,
    sp.FabI: 1, sp.TesA: 10, sp.FabF: 1, sp.FabB: 1, sp.FabA: 1,
}
for idx, conc in enzyme_concs.items():
    y0[idx] = float(conc)

SCALING_GROUPS = {g: 1 for g in scaling_groups}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

time_range = [0, 720]

generated_species = ()
generated_species_pattern = r"^C(\d+)_FA(_unsat)?$"

target_species = choose_species_indices(
    selected_species=generated_species,
    species_pattern=generated_species_pattern,
)

df_filtered = export_species_time_course(time_range, y0, theta, target_species, enzyme_name, save_csv=True)

# Initial conc vs final conc sweep: all nine enzymes + substrates (cofactors held fixed), 9 points, none at base
df_res = sweep_final_conc(
    make_nonuniform_sweep(y0, ["FabD", "FabH", "FabG", "FabZ", "FabI", "TesA", "FabF", "FabA", "FabB", "C3_MalCoA", "ACP", "C2_AcCoA"]),
    target_species, y0, theta, enzyme_name, max_steps=290
)

241 accepted steps, 268 total steps.
Saved to ../Data/Test_FullFAS/time_vs_conc.csv
Run complete!
241 accepted steps, 268 total steps.
Solver did not converge within 290 steps.
Skipping bad-condition row: {'FabD': 0.9, 'FabH': 1.0, 'FabG': 0.9, 'FabZ': 0.9, 'FabI': 1.0, 'TesA': 9.0, 'FabF': 1.0, 'FabA': 0.9, 'FabB': 0.9, 'C3_MalCoA': 500.0, 'ACP': 9.0, 'C2_AcCoA': 500.0}
257 accepted steps, 289 total steps.
Skipping too-similar row (< 0.2 log10 units from an already-kept row): {'FabD': 1.0, 'FabH': 1.0, 'FabG': 1.0, 'FabZ': 1.0, 'FabI': 1.0, 'TesA': 10.0, 'FabF': 1.0, 'FabA': 1.0, 'FabB': 1.0, 'C3_MalCoA': 600.0, 'ACP': 10.0, 'C2_AcCoA': 700.0}
Solver did not converge within 290 steps.
Skipping bad-condition row: {'FabD': 0.8, 'FabH': 0.7, 'FabG': 0.8, 'FabZ': 0.9, 'FabI': 0.8, 'TesA': 8.0, 'FabF': 0.7, 'FabA': 0.8, 'FabB': 0.9, 'C3_MalCoA': 400.0, 'ACP': 8.0, 'C2_AcCoA': 400.0}
Solver did not converge within 290 steps.
Skipping bad-condition row: {'FabD': 2.0, 'FabH': 1.0, 'FabG': 1.0